### Bronze Profile Summary:

Profile Summary script for the Bronze layer has been enhanced with advanced profiling metrics. This script identifies data quality issues such as null distributions and cardinality directly within the audit schema.

In [0]:
import json
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, TimestampType
from datetime import datetime

# --- 1. CONFIGURATION ---
# This section handles environment-specific parameters and defines where audit logs are stored.
try:
    # Use Databricks Widgets to accept a JSON array of dataset names as an input parameter.
    # This makes the notebook reusable for different pipeline runs.
    dbutils.widgets.text("datasets_json", "[]", "Datasets JSON Array")
    raw_input = dbutils.widgets.get("datasets_json")
    
    # Define constants for the source data location and the target audit storage.
    SOURCE_CATALOG = "data_bronze"
    SOURCE_SCHEMA  = "bronze"
    AUDIT_SCHEMA   = "audit"
    # The profile_summary table serves as the permanent history of data quality runs.
    AUDIT_TABLE    = f"{SOURCE_CATALOG}.{AUDIT_SCHEMA}.profile_summary"
except Exception as e:
    print(f"Setup failed: {str(e)}")
    raise

# --- 2. MODULAR FUNCTIONS ---

def ensure_audit_table():
    """
    Validates that the target audit table exists. 
    If missing, it creates it using Delta Lake with explicit schema definitions.
    """
    try:
        if not spark.catalog.tableExists(AUDIT_TABLE):
            print(f"Initializing Audit Table: {AUDIT_TABLE}")
            # Explicit DDL ensures the table is created with the correct long-term storage types.
            spark.sql(f"""
                CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
                    dataset_name STRING,
                    layer STRING,
                    row_count LONG,
                    column_count LONG, -- Stored as LONG to match Spark's default integer type
                    null_count LONG,
                    null_percent DOUBLE,
                    unique_count LONG,
                    columns STRING,
                    profile_timestamp TIMESTAMP
                ) USING DELTA
            """)
            # Adding table comments aids in Data Catalog discoverability.
            spark.sql(f"COMMENT ON TABLE {AUDIT_TABLE} IS 'Audit log for Bronze layer metrics.'")
    except Exception as e:
        print(f"Critical Error creating audit table: {str(e)}")
        raise

def profile_bronze_table(dataset_name):
    """
    Core logic: Calculates row counts, null distribution, and row uniqueness 
    for a specific Bronze table.
    """
    try:
        # Construct the path to the Bronze table in the catalog.
        full_table_name = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{dataset_name.lower()}"
        df = spark.table(full_table_name)
        
        # 1. METRIC CALCULATION: Basic Counts
        total_rows = df.count()
        num_cols = len(df.columns)
        
        # 2. METRIC CALCULATION: Null Density Analysis
        # We perform a single-pass scan of the DataFrame to find nulls in every column.
        null_counts_df = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
        total_nulls = sum(null_counts_df.first().asDict().values())
        
        # Calculate the overall null percentage of the dataset (Cells null / Total cells).
        null_percent = (total_nulls / (total_rows * num_cols)) * 100 if total_rows > 0 else 0.0
        
        # 3. METRIC CALCULATION: Uniqueness
        # Counts distinct rows to identify potential data duplication at the source.
        unique_count = df.distinct().count()
        
        # 4. DATA LOGGING: Package metrics into a metadata object.
        metadata = {
            "dataset_name": dataset_name,
            "layer": "BRONZE",
            "row_count": total_rows,
            "column_count": num_cols,
            "null_count": total_nulls,
            "null_percent": round(null_percent, 2),
            "unique_count": unique_count,
            "columns": ", ".join(df.columns),
            "profile_timestamp": datetime.now()
        }
        
        # 5. SCHEMA SAFETY: Explicitly define the Spark schema for the log record.
        # This prevents 'DELTA_MERGE_INCOMPATIBLE_DATATYPE' errors during the write.
        schema = StructType([
            StructField("dataset_name", StringType(), True),
            StructField("layer", StringType(), True),
            StructField("row_count", LongType(), True),
            StructField("column_count", LongType(), True),
            StructField("null_count", LongType(), True),
            StructField("null_percent", DoubleType(), True),
            StructField("unique_count", LongType(), True),
            StructField("columns", StringType(), True),
            StructField("profile_timestamp", TimestampType(), True)
        ])
        
        # 6. PERSISTENCE: Append the single-row profile record to the audit table.
        try:
            spark.createDataFrame([metadata], schema=schema) \
                 .write.format("delta") \
                 .mode("append") \
                 .saveAsTable(AUDIT_TABLE)
            print(f"Profiled: {dataset_name} | Null%: {metadata['null_percent']}% | Uniq: {unique_count}")
        except Exception as write_err:
            print(f"Write failed for {dataset_name}. Potential schema mismatch: {str(write_err)}")
        
    except Exception as e:
        print(f"Profiling calculation failed for {dataset_name}: {str(e)}")

# --- 3. ORCHESTRATION ---

def run_bronze_profiling():
    """
    Main Batch Orchestrator: Loops through all datasets provided in the widget.
    """
    try:
        # Step A: Ensure the landing table for audit logs is ready.
        ensure_audit_table()
        
        # Step B: Deserialize the dataset array from the widget.
        dataset_list = json.loads(raw_input)
        
        if not dataset_list:
            print("No datasets found in input parameters.")
            return

        # Step C: Iterate and profile each dataset sequentially.
        for ds in dataset_list:
            profile_bronze_table(ds)
            
        print(f"\n[SUCCESS] Bronze profiling complete. Metrics logged to {AUDIT_TABLE}.")
    except Exception as e:
        print(f"Batch Orchestration failed: {str(e)}")
        raise e

if __name__ == "__main__":
    run_bronze_profiling()

### Unit Testing & Evidence Collection
As per the Deliverable Standards, you must collect evidence of the profiling results.

In [0]:
import json
from pyspark.sql import functions as F

# --- 1. CONFIGURATION & PARAMETERS ---
# This section ensures the notebook can accept dynamic inputs for batch processing.
try:
    # Initialize the Databricks widget to accept a JSON array (e.g., ["city_time_series", "zip_time_series"])
    dbutils.widgets.text("datasets_json", "[]", "Datasets JSON Array")
    # Retrieve the string input from the widget
    raw_input = dbutils.widgets.get("datasets_json")
    
    # Standard location for the audit logs in the Bronze Catalog
    AUDIT_TABLE = "data_bronze.audit.profile_summary"
except Exception as e:
    print(f"Test Configuration failed: {str(e)}")
    raise

# --- 2. MODULAR TEST FUNCTION ---

def test_profile_metrics_validity(dataset_name):
    """
    Validates that the profiling metrics for a specific dataset are mathematically 
    and logically consistent. This prevents erroneous reporting of data health.
    """
    try:
        print(f"--- Validating Profiling Metrics: {dataset_name} ---")
        
        # 1. FETCH LATEST RECORD: Isolate the most recent profile run for the dataset.
        # We sort by the profile_timestamp in descending order and take the top record.
        audit_df = spark.table(AUDIT_TABLE)
        latest_metrics = (audit_df
                          .filter(F.col("dataset_name") == dataset_name)
                          .orderBy(F.col("profile_timestamp").desc())
                          .limit(1))
        
        # Ensure the dataset has actually been profiled at least once
        if latest_metrics.count() == 0:
            raise AssertionError(f"No profiling data found in {AUDIT_TABLE} for {dataset_name}")

        # Extract the metrics for validation
        metrics = latest_metrics.select("null_percent", "row_count", "unique_count").first()
        
        # 2. VALIDATION: Null Percentage Boundary
        # A null percentage must logically be between 0 and 100.
        null_p = metrics['null_percent']
        assert 0 <= null_p <= 100, f"FAILED: Null percent {null_p}% out of bounds for {dataset_name}"
        
        # 3. VALIDATION: Row Count Sanity 
        # Prevents technical errors where row counts are reported as negative values.
        rows = metrics['row_count']
        assert rows >= 0, f"FAILED: Negative row count detected for {dataset_name}"
        
        # 4. VALIDATION: Uniqueness Logic Check
        # The number of unique rows cannot mathematically exceed the total number of rows.
        # If this fails, it indicates a calculation error in the profiling logic.
        uniq = metrics['unique_count']
        assert uniq <= rows, f"FAILED: Unique count {uniq} exceeds total rows {rows} for {dataset_name}"
        
        print(f"PASSED: {dataset_name} | Null%: {null_p}% | Rows: {rows} | Unique: {uniq}")

    except Exception as e:
        # Standardize the error output for debugging in the Databricks Job console.
        print(f"Profiling Test Failed for {dataset_name}: {str(e)}")
        raise

# --- 3. ORCHESTRATION ---

def run_profiling_unit_tests():
    """
    Main Batch Orchestrator: Iterates through the provided list of datasets 
    and executes the validation logic for each.
    """
    try:
        # Deserialize the JSON string into a Python list object
        dataset_list = json.loads(raw_input)
        
        if not dataset_list:
            print("No datasets provided for profiling validation.")
            return

        # Execute tests sequentially for each dataset in the list
        for ds in dataset_list:
            test_profile_metrics_validity(ds)
            
        print(f"\n[SUCCESS] All profiling metrics verified for Bronze Layer.")

    except Exception as e:
        # Catch JSON parsing errors or batch-level failures
        print(f"Batch Profiling Test Execution Error: {str(e)}")
        raise e

if __name__ == "__main__":
    # Entry point for the execution
    run_profiling_unit_tests()